In [ ]:
# @title Setup (Kaggle / Colab) — clone repo + cài dependency
from pathlib import Path
import os, subprocess, sys

REPO = "https://github.com/MinMinhMin/Day21-Track3-Finetuning-Lab"
if (Path.cwd() / "scripts" / "colab_run.py").is_file():
    REPO_DIR = Path.cwd()
else:
    if Path("/kaggle/working").is_dir():
        WORK_DIR = Path("/kaggle/working")
    elif Path("/content").is_dir():
        WORK_DIR = Path("/content")
    else:
        WORK_DIR = Path.cwd()
    REPO_DIR = WORK_DIR / "Day21-Track3-Finetuning-Lab"
    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").exists():
            raise RuntimeError(f"Đã có thư mục nhưng không phải git repo: {REPO_DIR}")
        subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))

# Install from requirements.txt so the Kaggle and Colab environments use the same pins.
# Their preinstalled CUDA torch is retained when it already satisfies the requirement.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

os.environ.setdefault("COMPUTE_TIER", "T4")
import torch
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("Repo:", REPO_DIR)
print("GPU(s):", gpu_names if gpu_names else "NONE — bật GPU trong Notebook Settings")


# NB6 (tuỳ chọn) — Merge, kiểm chứng sau merge, và hoán đổi adapter

Deck §23. Hai đường triển khai:
* **Merge** — `W = W₀ + (α/r)·BA`, đồ thị phục vụ giống hệt base → **không** overhead.
* **Giữ riêng** — một base trong VRAM, nhiều adapter, chọn theo từng request.

> Ngày 20 sở hữu tầng phục vụ (vLLM/SGLang). Ở đây chỉ làm phần **quyết định lúc huấn
> luyện ảnh hưởng gì tới lúc phục vụ** — rồi bàn giao.

In [ ]:
import json, os, pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from labkit import evaluate as ev, generate, report
from labkit.config import get_tier

ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / "data").exists() else pathlib.Path.cwd().parent
TIER = get_tier(os.environ.get("COMPUTE_TIER", "T4"))

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

# EVAL_LIMIT=0 (unset) means the FULL set here, exactly as it does in NB2 and NB5.
# It used to default to 20, so an unabridged run silently scored this notebook's
# no-regression assert on 20 of 50 items while every other notebook used all of them.
EVAL_LIMIT = int(os.environ.get("EVAL_LIMIT", "0") or 0)
target = load_jsonl(ROOT / "data" / "eval_target.jsonl")
if EVAL_LIMIT:
    target = target[:EVAL_LIMIT]
print(f"scoring merge check on {len(target)} target items")

## 1. Điểm TRƯỚC merge

In [ ]:
from peft import PeftModel

model, tok = generate.load_base(TIER)
model = PeftModel.from_pretrained(model, str(ROOT / "adapters" / "correct"))
preds, _ = generate.generate_batch(model, tok, [r["input"] for r in target],
                                   system=generate.NAIVE_PROMPT)
before = sum(ev.triage_field_accuracy(p, r["label"]) for p, r in zip(preds, target)) / len(target)
print(f"trước merge: {before:.4f}")

## 2. Merge và đo lại

**Assert bắt buộc.** Merge là phép toán chính xác về mặt lý thuyết, nhưng dtype khi
gộp có thể làm tụt điểm. Nếu tụt, đừng deploy — hãy tìm hiểu vì sao.

In [ ]:
merged = model.merge_and_unload()
preds_m, _ = generate.generate_batch(merged, tok, [r["input"] for r in target],
                                     system=generate.NAIVE_PROMPT)
after = sum(ev.triage_field_accuracy(p, r["label"]) for p, r in zip(preds_m, target)) / len(target)
delta = after - before
print(f"sau merge:   {after:.4f}   (Δ {delta:+.4f})")

TOL = 0.01
assert delta >= -TOL, (
    f"điểm TỤT {abs(delta):.4f} sau merge (ngưỡng {TOL}). Kiểm tra dtype lúc merge; "
    "với DoRA cần PEFT ≥ 0.10 để gộp đúng vector magnitude (deck §23)."
)

out = ROOT / "adapters" / "merged"
merged.save_pretrained(out); tok.save_pretrained(out)
report.write_json({"before_merge": before, "after_merge": after, "delta": delta,
                   "tolerance": TOL, "n": len(target)},
                  "merge_check.json", results_dir=ROOT / "results")
del merged; generate.free_memory()

## 3. Một base, nhiều adapter — hoán đổi theo request

Đây là lập luận kinh tế của LoRA ở deck §23: base nằm trong VRAM một lần, mỗi khách
hàng/tác vụ là một adapter vài chục MB.

In [ ]:
model, tok = generate.load_base(TIER)
model = PeftModel.from_pretrained(model, str(ROOT / "adapters" / "correct"),
                                  adapter_name="correct")
available = ["correct"]
for extra in ("attn_only", "qlora"):
    d = ROOT / "adapters" / extra
    if d.exists():
        model.load_adapter(str(d), adapter_name=extra)
        available.append(extra)

print("adapter đang nạp:", available)
ticket = target[0]["input"]
for name in available:
    model.set_adapter(name)
    out, _ = generate.generate_batch(model, tok, [ticket], system=generate.NAIVE_PROMPT)
    print(f"\n[{name}] -> {out[0][:140]}")

## ✅ Checkpoint NB6
- [ ] `results/merge_check.json` — điểm sau merge không tụt quá 0.01
- [ ] Đã hoán đổi ≥2 adapter trên **cùng một** base đang nạp

→ Ngày 20 (Model Serving) là nơi biến việc này thành một endpoint đa người thuê.

## 4. Đóng gói toàn bộ repo và tải ZIP lên Hugging Face

Trên Kaggle, thêm Secret tên `HF_TOKEN` với quyền ghi vào repo Hugging Face `MinMinMinMin/KL` trước khi chạy ô kế tiếp (Notebook → Add-ons → Secrets). ZIP chứa source, data, notebooks, adapters và results; loại `.env`, file token, `__pycache__`, `.git`, virtualenv và cache. Archive được tạo trong `/kaggle/working` bên ngoài checkout repo.


In [ ]:
# @title 4. Tạo ZIP repo (không gồm secrets/cache) và upload lên Hugging Face
from pathlib import Path
import os
import zipfile

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login

HF_REPO_ID = "MinMinMinMin/KL"
HF_REPO_TYPE = "model"
HF_REPO_SUBDIR = "lab21_repo"

ROOT = Path.cwd()
if not (ROOT / "results").is_dir():
    raise FileNotFoundError(f"Không thấy results/ trong repo: {ROOT}")

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Kaggle Secret 'HF_TOKEN' đang rỗng.")

workspace = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else ROOT.parent
zip_path = workspace / f"{ROOT.name}_results.zip"

excluded_dirs = {
    ".git", "__pycache__", ".venv", "venv", ".ipynb_checkpoints",
    ".pytest_cache", ".cache", "hf_cache",
}
file_count = 0
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for current, dirnames, filenames in os.walk(ROOT):
        dirnames[:] = [name for name in dirnames if name not in excluded_dirs]
        for filename in filenames:
            source = Path(current) / filename
            if source.resolve() == zip_path.resolve():
                continue
            if filename == ".env" or (filename.startswith(".env.") and filename != ".env.example"):
                continue
            if filename.endswith((".pyc", ".pyo", ".token")):
                continue
            relative = source.relative_to(ROOT)
            archive_name = Path(ROOT.name) / relative
            archive.write(source, archive_name.as_posix())
            file_count += 1

# Confirm the requested secret/cache exclusions before sending the archive.
with zipfile.ZipFile(zip_path) as archive:
    members = [Path(name) for name in archive.namelist()]
    bad_members = [
        member.as_posix() for member in members
        if any(part in excluded_dirs for part in member.parts)
        or member.name == ".env"
        or (member.name.startswith(".env.") and member.name != ".env.example")
        or member.suffix in {".pyc", ".pyo", ".token"}
    ]
if bad_members:
    zip_path.unlink(missing_ok=True)
    raise RuntimeError(f"ZIP chứa file bị loại trừ: {bad_members[:10]}")

print(f"ZIP: {zip_path} ({file_count} files, {zip_path.stat().st_size / 1024**2:.1f} MB)")

try:
    login(token=hf_token, add_to_git_credential=False)
    upload_path = f"{HF_REPO_SUBDIR.strip('/')}/{zip_path.name}"
    HfApi().upload_file(
        path_or_fileobj=str(zip_path),
        path_in_repo=upload_path,
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
    )
    print("Uploaded:", f"https://huggingface.co/{HF_REPO_ID}/blob/main/{upload_path}")
finally:
    del hf_token
